# AAI6610 Module 8 — Credit Risk Prediction Pipeline: Re-Evaluation & Hyperparameter Tuning

## Michael Zimmerman | March 2026

### Pipeline Overview
This notebook extends the credit risk prediction pipeline developed across Modules 3, 5, and 7.  
Building on the midterm assessment, this module:

1. **Re-evaluates** all four models (Logistic Regression, Random Forest, LightGBM, Neural Network) with expanded metrics: **Precision, Recall, and F1 Score**
2. **Addresses the class weight gap** identified in the midterm — statistical models now receive balanced class weights for a fair comparison
3. **Performs hyperparameter tuning** on the best-performing model (Neural Network) to optimize default detection
4. **Analyzes** what effect tuning had on model performance, training time, and convergence

**Dataset:** UCI Default of Credit Card Clients (30,000 records, 24 features)  
**Target:** Binary classification — default (1) vs. no default (0)


In [2]:
# ================ IMPORTS AND CONFIGURATION ================

# Core libaries 
import pandas as pd
import numpy as np
import random 
import time # instead of relying on the ipynb cell timer, can use this to track the time taken for each process 

print("All Core Libraries Imported!")

# Visualization libraries 
import matplotlib.pyplot as plt
import seaborn as sns

print("All Visualization Libraries Imported!")

# ------- Machine Learning Libraries ------

# Preprocessing and Splitting 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
print("All Preprocessing and Splitting Libraries Imported!")

# ML Models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
print("All ML Models Imported!")

# LightGBM
from lightgbm import LGBMClassifier
print("LightGBM Imported!")

# Metrics & Evaluation
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score,
    confusion_matrix, 
    classification_report,
    ConfusionMatrixDisplay
)
print("All Metrics and Evaluation Libraries Imported!")

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
print("TensorFlow and Keras Imported!")

# final confirmation statement 
print("=" * 50)
print("All Libraries Imported Successfully!")
print("=" * 50)
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")

All Core Libraries Imported!
All Visualization Libraries Imported!
All Preprocessing and Splitting Libraries Imported!
All ML Models Imported!
LightGBM Imported!
All Metrics and Evaluation Libraries Imported!
TensorFlow and Keras Imported!
All Libraries Imported Successfully!
TensorFlow version: 2.20.0
NumPy version:      2.4.1
Pandas version:     2.3.3


# GLOBAL CONFIGURATION


In [3]:
'''
RANDOM_STATE = 2020 
This will be used consistently across all processes

Other notebooks used RANDOM_STATE = 2020 for splitting and RANDOM_SEED = 2026 for TF/NumPy
Consilidaitng it to a single value will keep things simple, clean, consistent. 

Operations that will use it:
- train_test_split
- sklearn model instantiations 
- NumPy and TensorFlow random seeds 
- sampling for qualitative analysis 

'''


# Random state and test size config
RANDOM_STATE = 2020
TEST_SIZE = 0.2

# Set random seeds for reproducibility
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# display config for readability 
pd.set_option('display.max_columns', None) # show all columns in DataFRame output
pd.set_option('display.width', 120) # set display width for better readability

print("Configuration Set: Random State, Test Size, and Display Options")


Configuration Set: Random State, Test Size, and Display Options


# Data Loading & Preprocessing

In [4]:
'''
Reusing the same structure for data loading and preprocesssing from module 3 and 5 
combining into a single section and using the same:
 - dataset 
 - split 
 - scaling 
'''

# DATA PROCESSING & EXPLORATION 
# ==============================

# Load the dataset 
# read the csv file into a dataframe
df = pd.read_csv('./dataset/default_of_credit_card_clients.csv')

print("-" * 50)
print("Dataset loaded successfully!")
print(f"\nData shape: {df.shape}")
print("-" * 50)

#========= Target Distribution ============
target_distribution = df['default payment next month'].mean()
print(f'Target Distribution (default rate): {target_distribution:.4f} ({target_distribution*100:.2f}%)')






--------------------------------------------------
Dataset loaded successfully!

Data shape: (30000, 25)
--------------------------------------------------
Target Distribution (default rate): 0.2212 (22.12%)


In [5]:
#========= Data Cleaning ===========

# cleaning EDUCATION Feature 
'''
Education had some invalid values beyond the expected 1-4 categories:
    - 0, 5, and 6 were all recoded to 4 "Other"
    - Expected categories:
        1: Graduate School
        2: University
        3: High School
        4: Other (including the invalid values)
'''
df.iloc[df['EDUCATION'].isin([0, 5, 6]), df.columns.get_loc('EDUCATION')] = 4


# cleaning MARRIAGE Feature
'''
Marriage had some invalid values like 0 which were recoded to 3 "Other"
Expected categories: 
    1: Married
    2: Single
    3: Other (including the invalid value)
mapping 0 -> 3
'''
df.iloc[df['MARRIAGE'].isin([0]), df.columns.get_loc('MARRIAGE')] = 3


print(" Education and Marriage features cleaned successfully!")
print("-" * 50)
print(f'Cleaned EDUCATION values: {df["EDUCATION"].unique()}')
print(f'Cleaned MARRIAGE values: {df["MARRIAGE"].unique()}')

 Education and Marriage features cleaned successfully!
--------------------------------------------------
Cleaned EDUCATION values: [2 1 3 4]
Cleaned MARRIAGE values: [1 2 3]


In [6]:
#========= Feature and Target Setup ============


# Target and Drop column from dataset 
TARGET_COLUMN = 'default payment next month'
DROP_COLUMNS = ['ID' , 'default payment next month'] 


# create X (features) and y (target)
y = df[TARGET_COLUMN].values
X = df.drop(columns=DROP_COLUMNS).values

# store features names for later for interpretability and visualization use
df_features = df.drop(columns=DROP_COLUMNS).columns.tolist()

# check updated shapes with newly dropped columns
print("Feature and Target Shapes: \n")
print(f'Features Shape: {X.shape}')
print(f'Target Shape: {y.shape}')
print("-" * 50)
print(f"Number of features: {len(df_features)}")
print(f"Feature Names: {df_features[:5]}") # print the first 5 features

print("=" * 70)

Feature and Target Shapes: 

Features Shape: (30000, 23)
Target Shape: (30000,)
--------------------------------------------------
Number of features: 23
Feature Names: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE']


In [7]:
#========= Train - Test - Split ============
# call RANDOM_STATE and TEST_SIZE to get exact split from assignment 3
X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=TEST_SIZE, 
            random_state=RANDOM_STATE
)

print("-" * 50)
print(f"Random state parameters : {RANDOM_STATE}")
print(f'Training Set Shape: {X_train.shape[0]:,} samples')
print(f'Test Set Shape: {X_test.shape[0]:,} samples')
print("-" * 50)
print(f'Number of positive samples in training set: {np.sum(y_train)}')
print(f'Number of positive samples in test set: {np.sum(y_test)}')
print(f'Training default rate: {np.mean(y_train):.4f}')
print(f'Training default rate: {np.mean(y_test):.4f}')
print("-" * 50)
print("Train/Test split completed successfully with the same parameters as Assignment 3!")


# ========== FEATURE SCALING ===========

'''
Neural networks require feature scaling for optimal performance
'''
# initialize StandardScaler for scaling 
scaler = StandardScaler()

# fit the scaler on the training data then transform
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

print(f'Training Set Shape: {X_train_scaled.shape}')
print(f'Test Set Shape: {X_test_scaled.shape}')


# ========== Class Weights ===========
'''
Addressing the 22.1% class imbalance 
All models will be configured for balanced classes 
'''
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train), # returns an assorted array of unique class labels
    y=y_train
)

# creating a dictionary to pass to the Keras model and to map and penalize mistakaes in the default class more heavily during training. Required for Keras API
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("-" * 50)
print("Class weights computed successfully!\n")
print(f"Non-Default (Class 0) weight: {class_weight_dict[0]:.4f}")
print(f"Default (Class 1) weight: {class_weight_dict[1]:.4f}")
print(f"Ratio: {class_weight_dict[1]/class_weight_dict[0]:.4f}")

print("=" * 65)
print("Data loading, cleaning, preprocessing, and splitting completed successfully!")

--------------------------------------------------
Random state parameters : 2020
Training Set Shape: 24,000 samples
Test Set Shape: 6,000 samples
--------------------------------------------------
Number of positive samples in training set: 5271
Number of positive samples in test set: 1365
Training default rate: 0.2196
Training default rate: 0.2275
--------------------------------------------------
Train/Test split completed successfully with the same parameters as Assignment 3!
Training Set Shape: (24000, 23)
Test Set Shape: (6000, 23)
--------------------------------------------------
Class weights computed successfully!

Non-Default (Class 0) weight: 0.6407
Default (Class 1) weight: 2.2766
Ratio: 3.5532
Data loading, cleaning, preprocessing, and splitting completed successfully!


# BASELINE RESULTS FROM MODULE 3 & 5 

In [8]:
'''
From the original results from the past modules the additional performance metrics (F1 score, precision, recall)
were calculated but output was never included in final comparison table 

This section here will hard code that data instead of importing from each notebook. Reason for when graded the original notebooks will not be available to import


Some key findings and items that need to be addressed:

- Statistical Models: no class weights, default hyperparameters
- Neural Network: class weights applied (class_weight_dict), layer+hidden layer architechture 64>32>16>1

Since statistical models never received class weights or tuning it contributed to massive recall gap of 32% between the NN (60.59%)

'''

baseline_results = {
    'Logistic Regression': {
        'accuracy': 0.8122,
        'roc_auc': 0.7249,
        'recall': 0.2366,
        'precision': 0.7917,
        'f1-score': 0.3644,
        'class_weights': False
    },
    'Random Forest': {
        'accuracy': 0.8223,
        'roc_auc': 0.7727,
        'recall': 0.3707,
        'precision': 0.7097,
        'f1-score': 0.4870,
        'class_weights': False
    },
    'LightGBM': {
        'accuracy': 0.8222,
        'roc_auc': 0.7889,
        'recall': 0.3626,
        'precision': 0.7153,
        'f1-score': 0.4813,
        'class_weights': False
    },
    'Neural Network': {
        'accuracy': 0.7663,
        'roc_auc': 0.7839,
        'recall': 0.6059,
        'precision': 0.4891,
        'f1-score': 0.5412,
        'class_weights': True
    }
}


print("Baseline results created successfully!")
print("-" * 50)
print("Baseline Results Summary:")
baseline_df = pd.DataFrame(baseline_results).T # transpose to have models as rows and metrics as columns
baseline_df.index.name = 'Model'
print(baseline_df)


Baseline results created successfully!
--------------------------------------------------
Baseline Results Summary:
                    accuracy roc_auc  recall precision f1-score class_weights
Model                                                                        
Logistic Regression   0.8122  0.7249  0.2366    0.7917   0.3644         False
Random Forest         0.8223  0.7727  0.3707    0.7097    0.487         False
LightGBM              0.8222  0.7889  0.3626    0.7153   0.4813         False
Neural Network        0.7663  0.7839  0.6059    0.4891   0.5412          True


# THE PIPELINE CLASS ARCHITECTURE
### Combining Module 3 and 5 implementation 

In [9]:
'''
Redesigning the pipeline and tuning process to address the following key issues:
- lack of class weights for statistical models
- lack of hyperparameter tuning for all models

Here to support the redesign:
- pre-split data, all models share the same train-test-split for consistency and comparability
- both sklearn and keras in one unified class
- expanded evaluation metrics including precision, recall, f1-score on default class (1)
    - will address gaps and imbalance issues more directly
- training time tracking for each model, instead of using cell timers. More accurate and consistent
    - will help with hyperparameter tuning comparison and understanding trade-offs

'''
class CreditRiskPipeline:
    def __init__(self, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, df_features):
        '''
            - accepts pre-split data instead of splitting
                - models will be trained on the same data
                - all models will use the same split

            Params:
                - X_train, X_test : unscaled features (for tree based models that do not require scaling)
                - y_train, y_test : target arrays
                - df_features : scaled features (for LR and NN, require scaling)
        '''

        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test

        # scaled features for LR and NN
        self.X_train_scaled = X_train_scaled
        self.X_test_scaled = X_test_scaled
        self.df_features = df_features

        # initialize empty dictionaries to store models and performance metrics results
        self.models = {}
        self.results = {}  # will be used to store evaluation metrics for each model

        print("Pipeline Initialized Successfully!")
        print("-" * 50)
        print(f"Training Set Shape: {self.X_train.shape[0]} samples")
        print(f"Test Set Shape: {self.X_test.shape[0]} samples")
        print(f"Features: {len(self.df_features)}")

    def train_model(self, model, model_name, scale=False, batch_size=32):
        '''
        Train all models using the same train-test-split + all 6 metrics

        Params:
            - model: sklearn or keras model instance
            - model_name: model string identifier
            - scale: True for models that need scaling (LR & NN), default set to False for tree based models
        '''
        # selecting the appropriate data based on scaling requirement
        if scale:
            X_train_data = self.X_train_scaled
            X_test_data = self.X_test_scaled
        else:
            X_train_data = self.X_train
            X_test_data = self.X_test

        # ============= Training Time Tracking - START =============
        # track training time for each model
        start_time = time.time()
        training_time = time.time() - start_time

        # ============== TRAINING ALL MODELS ==============

        # if the model is a sklearn model (LR, RF, LGBM)
        if isinstance(model, (LogisticRegression, RandomForestClassifier, LGBMClassifier)):
            model.fit(
                X_train_data,
                self.y_train,
            )  
        # if the model is a Keras Sequential model (NN)
        if isinstance(model, Sequential):
            early_stop = EarlyStopping(
                monitor="val_loss",
                patience=25,
                restore_best_weights=True,
                verbose=1
            )

            model.fit(
                X_train_data,
                self.y_train,
                epochs=100,                      # high epochs helps early stopping find optimal duration
                batch_size=batch_size,                   # number of samples per gradient update
                validation_split=0.2,            # use 20% of training data for validation
                class_weight=class_weight_dict,  # pass class weights to handle class imbalance
                callbacks=[early_stop],
                verbose=0
            )
        
        # ============= Training Time Tracking - END, needs to be after fit() =============
        # track training time for each model
        training_time = time.time() - start_time
        
        # ========= Predictions and Probailities ============
        if isinstance(model, Sequential):
            y_probabilities = model.predict(X_test_data, verbose=0).flatten()   # flatten to convert from 2D array to 1D
            y_predictions = (y_probabilities >= 0.5).astype(int)                # thresholding at 0.5 for binary classification   
        else:
            y_predictions = model.predict(X_test_data)                          # make predictions on the test set 
            y_probabilities = model.predict_proba(X_test_data)[:, 1]            # get the predicted probabilities for the positive class (class 1)
        
        # calculate evaluation metrics
        accuracy = accuracy_score(self.y_test, y_predictions)
        roc_auc = roc_auc_score(self.y_test, y_probabilities)
        
        # Extract Evaluation Metrics from baseline_results and format for classification report
        
        #====== Classfication Report and Confusion Matrix =======
        print("=" * 50)
        print(f"Classfication Report for {model_name}")
        class_report = classification_report(
            self.y_test, 
            y_predictions,
            target_names=['No Default (0)', 'Default (1)'],
            digits=4,
            output_dict=True
        )
        
        # retrieve all Default metrics (class 1) only
        default_metrics = class_report['Default (1)']
        
        # store results in self.results, created to store evaluation metrics for each model
        self.results[model_name] = {
            'predictions': y_predictions,
            'probabilities': y_probabilities,
            'accuracy': accuracy,
            'roc_auc': roc_auc,
            'recall': default_metrics['recall'],
            'precision': default_metrics['precision'],
            'f1-score': default_metrics['f1-score'],
            'training_time': training_time
        }
        
        # store models in self.models for reference and predictions 
        self.models[model_name] = {
            'model': model,
            'scaled': scale
        }
        
        print("=" * 50)
        print(f'Trained in: {training_time:.2f} seconds')
        
        print("-" * 50)
        print(f'Accuracy: {self.results[model_name]["accuracy"]:.4f}')
        print(f'Model as a Percentage: {self.results[model_name]["accuracy"]*100:.1f}%')
        
        print("-" * 50)
        print(f'ROC AUC: {self.results[model_name]["roc_auc"]:.4f}')
        print(f'Model as a Percentage: {self.results[model_name]["roc_auc"]*100:.1f}%')
        
        print("-" * 50)
        print(f'Recall: {self.results[model_name]["recall"]:.4f}')
        print(f'Model as a Percentage: {self.results[model_name]["recall"]*100:.1f}%')
        
        print("-" * 50)
        print(f'Precision: {self.results[model_name]["precision"]:.4f}')
        print(f'Model as a Percentage: {self.results[model_name]["precision"]*100:.1f}%')
        
        print("-" * 50)
        print(f'F1 Score: {self.results[model_name]["f1-score"]:.4f}')
        print(f'Model as a Percentage: {self.results[model_name]["f1-score"]*100:.1f}%\n')
        
        
        
        
        
print("CreditRiskPipeline class defined successfully!")
print("-" * 50)

CreditRiskPipeline class defined successfully!
--------------------------------------------------


In [10]:
pipeline = CreditRiskPipeline(X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, df_features)
pipeline



# Testing train_model using Logistic Regression with scaling and class weights
lr_model = LogisticRegression(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    max_iter=1000,   
)
pipeline.train_model(lr_model, 'Logistic Regression', scale=True)

Pipeline Initialized Successfully!
--------------------------------------------------
Training Set Shape: 24000 samples
Test Set Shape: 6000 samples
Features: 23
Classfication Report for Logistic Regression
Trained in: 0.21 seconds
--------------------------------------------------
Accuracy: 0.6963
Model as a Percentage: 69.6%
--------------------------------------------------
ROC AUC: 0.7255
Model as a Percentage: 72.6%
--------------------------------------------------
Recall: 0.6330
Model as a Percentage: 63.3%
--------------------------------------------------
Precision: 0.3954
Model as a Percentage: 39.5%
--------------------------------------------------
F1 Score: 0.4868
Model as a Percentage: 48.7%



# TRAIN BASELINE STATISTICAL MODELS W/CLASS WEIGHTS

In [11]:
# LOGISTIC REGRESSION
lr_model = LogisticRegression(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    max_iter=1000,   
)
pipeline.train_model(lr_model, 'Logistic Regression', scale=True)


# RANDOM FOREST 
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_estimators=100,   
)
pipeline.train_model(rf_model, 'Random Forest', scale=False)


# LIGHT GRADIENT BOOSTING MACHINE - LightGBM
lgbm_model = LGBMClassifier(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_estimators=100, 
    is_unbalanced=True,
    verbose=-1          # suppress LightGBM training output for cleaner display  
)
pipeline.train_model(lgbm_model, 'LightGBM', scale=False)

Classfication Report for Logistic Regression
Trained in: 0.04 seconds
--------------------------------------------------
Accuracy: 0.6963
Model as a Percentage: 69.6%
--------------------------------------------------
ROC AUC: 0.7255
Model as a Percentage: 72.6%
--------------------------------------------------
Recall: 0.6330
Model as a Percentage: 63.3%
--------------------------------------------------
Precision: 0.3954
Model as a Percentage: 39.5%
--------------------------------------------------
F1 Score: 0.4868
Model as a Percentage: 48.7%

Classfication Report for Random Forest
Trained in: 4.25 seconds
--------------------------------------------------
Accuracy: 0.8188
Model as a Percentage: 81.9%
--------------------------------------------------
ROC AUC: 0.7766
Model as a Percentage: 77.7%
--------------------------------------------------
Recall: 0.3399
Model as a Percentage: 34.0%
--------------------------------------------------
Precision: 0.7138
Model as a Percentage: 71

/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# TRAIN BASELINE NEURAL NETWORK

In [12]:
'''
This section will be used to build out the deep learning model architechture. 
Then compile the model with the appropriate loss function, optimizer and evaluation metrics

Using activation functions, dropout and early stopping to prevent overfitting and improve generalization
    - 1 x input layer (# of features)
    - 3 x hidden layers with ReLU activation and dropout for regularization, may had Leaky ReLU if performancr is poor and gradient is vanishing
    - 1 x outpput ;ayer with sigmoid activation for binary classification
ReLU activation for hidden layers. 
    Both activation functions are non-linear, allowing the model to learn from complex patterns within the hidden layers

Will be using Tensorflow/Keras for the build out 
Will use Sequential API for linear layers


- Decreasing hidden layers neurons (64>32>16) to specify the layer pattern size for the 3 layers 

- Dense: Z = (input x weights) + bias (linear math)
Then ReLU and Sigmoid is applied after the linear math is done

- Relu is the result is negative, turn it to 0, if positive keep the same value
- Sigmoid, squash the output to a value between 0 and 1


Resources:
https://keras.io/guides/sequential_model/
'''

nn_model = Sequential([
    # Input + First Hidden Layer, 23 is used because there are 23 features 
    Dense(64, activation='relu', name='hidden_layer_1', input_shape=(23,)),
    Dropout(0.3, name='dropout_1'),
    
    # Second Hidden Layer 
    Dense(32, activation='relu', name='hidden_layer_2'),
    Dropout(0.3, name='dropout_2'),
    
    # Third Hidden Layer
    Dense(16, activation='relu', name='hidden_layer_3'),
    Dropout(0.2, name='dropout_3'),
    
    # Output Layer 
    Dense(1, activation='sigmoid', name='output_layer')  
])


# =========== Compile the model ==========
'''
- uses  binary cross-entropy for loss function
    loss function handles binary classification problems and meassures between predicted and actual probabilities, which is appropriate for this credit risk prediction task.
- Adam optimizer for training 
- metrics for evaluation

Resources: https://keras.io/api/metrics/
'''
# used for metrics, optimzier, and loss function for configuration and not compute 
nn_model.compile(
    optimizer=Adam(learning_rate=0.001),            # adam with a learning rate of 0.001 for efficient training and convergence
    loss='binary_crossentropy', 
    metrics=['accuracy']  
)


print ("-" * 50)
print('Optimzer, Loss Function and Metrics configured successfully!\n')
print(f"Optimizer: Adam with learning rate of {nn_model.optimizer.learning_rate.numpy():.4f}")
nn_model.summary()  # print the model summary to verify the architecture and number of parameters
print("-" * 50)

print("\n") # adding a extra line for cleaner formatting 
# training the neural network model using the pipeline's train_model method
pipeline.train_model(nn_model, 'Neural Network', scale=True)

--------------------------------------------------
Optimzer, Loss Function and Metrics configured successfully!

Optimizer: Adam with learning rate of 0.0010


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 64)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_3 (Dense)          │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,161 (16.25 KB)

 Trainable params: 4,161 (16.25 KB)

 Non-trainable params: 0 (0.00 B)

--------------------------------------------------


Epoch 67: early stopping
Restoring model weights from the end of the best epoch: 42.
Classfication Report for Neural Network
Trained in: 34.03 seconds
--------------------------------------------------
Accuracy: 0.7547
Model as a Percentage: 75.5%
--------------------------------------------------
ROC AUC: 0.7782
Model as a Percentage: 77.8%
--------------------------------------------------
Recall: 0.6227
Model as a Percentage: 62.3%
--------------------------------------------------
Precision: 0.4704
Model as a Percentage: 47.0%
--------------------------------------------------
F1 Score: 0.5359
Model as a Percentage: 53.6%



# EVALUATE AND COMPARE ALL BASELINE MODELS

In [13]:
# Compare all 4 models with expanded metrics and training time in summary table 

for model_name in pipeline.results:
    # get predictions for the current model from the pipeline results dictionary
    y_pred = pipeline.results[model_name]['predictions'] 
    
    print ("=" * 50)
    print(f"Classfication Report for {model_name}")
    c_report = classification_report(
        pipeline.y_test, 
        y_pred,
        target_names=['No Default (0)', 'Default (1)'],
        digits=4
    )
    # print the classification report, that used the same predictions and true labels for all models
    print(c_report)
    print ("~" * 50)
    
    
    # print the confusion matric for each model
    print(f'Confusion Matrix for {model_name}')
    # calculate the confusion matrix 
    conf_matrix = confusion_matrix(pipeline.y_test, y_pred)
    print(conf_matrix)

  


# 

Classfication Report for Logistic Regression
                precision    recall  f1-score   support

No Default (0)     0.8687    0.7150    0.7844      4635
   Default (1)     0.3954    0.6330    0.4868      1365

      accuracy                         0.6963      6000
     macro avg     0.6320    0.6740    0.6356      6000
  weighted avg     0.7610    0.6963    0.7167      6000

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Confusion Matrix for Logistic Regression
[[3314 1321]
 [ 501  864]]
Classfication Report for Random Forest
                precision    recall  f1-score   support

No Default (0)     0.8316    0.9599    0.8911      4635
   Default (1)     0.7138    0.3399    0.4605      1365

      accuracy                         0.8188      6000
     macro avg     0.7727    0.6499    0.6758      6000
  weighted avg     0.8048    0.8188    0.7932      6000

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Confusion Matrix for Random Forest
[[4449  186]
 [ 901  464]]
Classf

In [14]:
# ======= Comparison Table of new metrics ===========

# creating and empty list to store the results for each model to have a unified comparison table
comparison_results = []

for model_name, metrics in pipeline.results.items():
    comparison_results.append({
        'Model': model_name,
        'Accuracy': round(metrics['accuracy'], 4),
        'ROC AUC': round(metrics['roc_auc'], 4),
        'Recall': round(metrics['recall'], 4),
        'Precision': round(metrics['precision'], 4),
        'F1-Score': round(metrics['f1-score'], 4),
        f"{'Training Time (SECONDS)':>25}": f"{metrics['training_time']:.2f}"
    })
    
# create a DataFrame for the comparison table for better formatting and visualizatino
comparison_df = pd.DataFrame(comparison_results)

# sort the dataframe by F1-Score in descending order
comparison_df = comparison_df.sort_values(by="F1-Score", ascending=False)

print("=" * 50)
print("Model Performance Comparison")
print("=" * 50)

# display comparison table with no index
print(comparison_df.to_string(index=False))
print("-" * 80)
    

Model Performance Comparison
              Model  Accuracy  ROC AUC  Recall  Precision  F1-Score   Training Time (SECONDS)
           LightGBM    0.7653   0.7844  0.6352     0.4879    0.5519                      0.74
     Neural Network    0.7547   0.7782  0.6227     0.4704    0.5359                     34.03
Logistic Regression    0.6963   0.7255  0.6330     0.3954    0.4868                      0.04
      Random Forest    0.8188   0.7766  0.3399     0.7138    0.4605                      4.25
--------------------------------------------------------------------------------


In [15]:
# ======= Baseline vs Module 8 Comparison Tables (Split by Metric) ===========

'''
The baseline results are from the original notebook results 
Module 8 are the new results with the redesigned pipeline with class weights and hyperparameter tuning

Metrics are split into 3 separate comparison tables for clarity
'''

recall_results = []
precision_results = []
f1_results = []

for model_name, metrics in pipeline.results.items():
    # get new metrics from the pipeline
    get_new_recall = metrics['recall']
    get_new_precision = metrics['precision']
    get_new_f1 = metrics['f1-score']
    
    # get baseline metrics from previous notebooks, stored in baseline_results dictionary 
    get_baseline_recall = baseline_results[model_name]['recall']
    get_baseline_precision = baseline_results[model_name]['precision']
    get_baseline_f1 = baseline_results[model_name]['f1-score']
    
    # handle change calculations to show the improvement from baseline to new results
    recall_change = get_new_recall - get_baseline_recall
    precision_change = get_new_precision - get_baseline_precision
    f1_change = get_new_f1 - get_baseline_f1
    
    # append recall metrics
    recall_results.append({
        'Model': model_name,
        'Baseline': round(get_baseline_recall, 4),
        'Module 8': round(get_new_recall, 4),
        'Change': f"{recall_change:+.4f}"
    })
    
    # append precision metrics
    precision_results.append({
        'Model': model_name,
        'Baseline': round(get_baseline_precision, 4),
        'Module 8': round(get_new_precision, 4),
        'Change': f"{precision_change:+.4f}"
    })
    
    # append f1-score metrics
    f1_results.append({
        'Model': model_name,
        'Baseline': round(get_baseline_f1, 4),
        'Module 8': round(get_new_f1, 4),
        'Change': f"{f1_change:+.4f}"
    })

# create DataFrames for each metric
recall_df = pd.DataFrame(recall_results)
precision_df = pd.DataFrame(precision_results)
f1_df = pd.DataFrame(f1_results)

# display recall comparison
print("=" * 50)
print("Recall Comparison: Baseline vs Module 8")
print("=" * 50)
print(recall_df.to_string(index=False))
print("-" * 50)

# display precision comparison
print("=" * 50)
print("Precision Comparison: Baseline vs Module 8")
print("=" * 50)
print(precision_df.to_string(index=False))
print("-" * 50)

# display f1-score comparison
print("=" * 50)
print("F1-Score Comparison: Baseline vs Module 8")
print("=" * 50)
print(f1_df.to_string(index=False))
print("-" * 80)

Recall Comparison: Baseline vs Module 8
              Model  Baseline  Module 8  Change
Logistic Regression    0.2366    0.6330 +0.3964
      Random Forest    0.3707    0.3399 -0.0308
           LightGBM    0.3626    0.6352 +0.2726
     Neural Network    0.6059    0.6227 +0.0168
--------------------------------------------------
Precision Comparison: Baseline vs Module 8
              Model  Baseline  Module 8  Change
Logistic Regression    0.7917    0.3954 -0.3963
      Random Forest    0.7097    0.7138 +0.0041
           LightGBM    0.7153    0.4879 -0.2274
     Neural Network    0.4891    0.4704 -0.0187
--------------------------------------------------
F1-Score Comparison: Baseline vs Module 8
              Model  Baseline  Module 8  Change
Logistic Regression    0.3644    0.4868 +0.1224
      Random Forest    0.4870    0.4605 -0.0265
           LightGBM    0.4813    0.5519 +0.0706
     Neural Network    0.5412    0.5359 -0.0053
-----------------------------------------------------

# TUNING CONFIGURATIONS FOR NEURAL NETWORK

In [16]:
'''     
Neural Network show a slight improvement in recall, but mere decrease in precision and F1-score
    - most likely due to class weights already being applied at the baseline so improvement and decrease is minimal
    - overall it ranks between a high 2nd and low 1st place across all metrics, showing consistent performance across the board
    - making it the most balanced, but not best performing 
    - applying hyperparameter tuning may cause a tradeoff but could improve overall performance and become the highest performing model across all metrics
    
    
Baseline Architecture and Configuration Issues:
- 64 > 32 > 16 hidden layers with ReLU + sigmoid activation and dropout for regularization, with no prior hyperparameter tuning or optimization
- class weights were applied 
- Optimizer: Adam with a learning rate of 0.001
- batch size of 32 and early stopping with patience of 10 epochs (prevents overfitting)
- initializer: default initializer for Dense layers when using Keras is Glorot (Xavier uniform)

Will test against different:
- optimzers ( Adam, RMSProp, )
'''

# ====== 4 versions of Hyperparameter tuning ========
# creating a list with different configs for tuning
# naming conventions: Pytorch uses Xavier and Keras used Glorot, they are the same initializer 

tuning_configs = [
    {
        'name': 'Config 1: Wider Network',
        'layers': [128,64,32],
        'dropouts': [0.3, 0.3, 0.2],
        'optimizer': Adam(learning_rate=0.001),
        'batch_size': 32,
        'initializer': 'glorot_uniform'
    },
    {
        'name': 'Config 2: SGD+Momentum',
        'layers': [64,32,16],
        'dropouts': [0.3, 0.3, 0.2],
        'optimizer': tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
        'batch_size': 32,
        'initializer': 'glorot_uniform'
    },
    {
        'name': 'Config 3: RMSProp',
        'layers': [64,32,16],
        'dropouts': [0.3, 0.3, 0.2],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.0005),
        'batch_size': 32,
        'initializer': 'glorot_uniform'
    },
    {
        'name': 'Config 4: Kaiming + Wider + Lower Dropout',
        'layers': [128,64,32],
        'dropouts': [0.2, 0.2, 0.1],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.001),
        'batch_size': 64,
        'initializer': 'he_normal'
    },
]

print("-" * 50)
print("Tuning Configurations initialized successfully!\n")
print("-" * 50)

--------------------------------------------------
Tuning Configurations initialized successfully!

--------------------------------------------------


In [17]:
# loop through each config and train a new model with one of the specified hyperparameter configurations from the tuning_configs list
for config in tuning_configs:
    print("=" * 50)
    print(f"Testing {config['name']}")
    print("=" * 50)

    model = Sequential([
    Dense(config['layers'][0], activation='relu', kernel_initializer=config['initializer'], name='hidden_layer_1', input_shape=(23,)),
    Dropout(config['dropouts'][0], name='dropout_1'),
    
    Dense(config['layers'][1], activation='relu', kernel_initializer=config['initializer'], name='hidden_layer_2'),
    Dropout(config['dropouts'][1], name='dropout_2'),
    
    Dense(config['layers'][2], activation='relu', kernel_initializer=config['initializer'], name='hidden_layer_3'),
    Dropout(config['dropouts'][2], name='dropout_3'),
    
    Dense(1, activation='sigmoid', name='output_layer')
])
    
    # ========= COMPILE THE MODEL ========= 
    model.compile(
    optimizer=config['optimizer'],
    loss='binary_crossentropy',
    metrics=['accuracy']
)

    print('Model compiled successfully with the following configuration:')
    for key, value in config.items():
        if key != 'optimizer':
            print(f"{key.capitalize()}: {value}")
        else:
            print(f"Optimizer: {type(value).__name__} with learning rate {value.learning_rate.numpy():.4f}")
        

    # ========= TRAIN THE MODEL ========= 
    pipeline.train_model(model, f"Neural Network - {config['name']}", scale=True)
    
print("-" * 50)
print("\n\nTuning Configs created successfully!\n")
print("-" * 50)

Testing Config 1: Wider Network
Model compiled successfully with the following configuration:
Name: Config 1: Wider Network
Layers: [128, 64, 32]
Dropouts: [0.3, 0.3, 0.2]
Optimizer: Adam with learning rate 0.0010
Batch_size: 32
Initializer: glorot_uniform


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 52: early stopping
Restoring model weights from the end of the best epoch: 27.
Classfication Report for Neural Network - Config 1: Wider Network
Trained in: 28.90 seconds
--------------------------------------------------
Accuracy: 0.7613
Model as a Percentage: 76.1%
--------------------------------------------------
ROC AUC: 0.7776
Model as a Percentage: 77.8%
--------------------------------------------------
Recall: 0.6081
Model as a Percentage: 60.8%
--------------------------------------------------
Precision: 0.4806
Model as a Percentage: 48.1%
--------------------------------------------------
F1 Score: 0.5369
Model as a Percentage: 53.7%

Testing Config 2: SGD+Momentum
Model compiled successfully with the following configuration:
Name: Config 2: SGD+Momentum
Layers: [64, 32, 16]
Dropouts: [0.3, 0.3, 0.2]
Optimizer: SGD with learning rate 0.0100
Batch_size: 32
Initializer: glorot_uniform


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 92: early stopping
Restoring model weights from the end of the best epoch: 67.
Classfication Report for Neural Network - Config 2: SGD+Momentum
Trained in: 41.04 seconds
--------------------------------------------------
Accuracy: 0.7717
Model as a Percentage: 77.2%
--------------------------------------------------
ROC AUC: 0.7772
Model as a Percentage: 77.7%
--------------------------------------------------
Recall: 0.6007
Model as a Percentage: 60.1%
--------------------------------------------------
Precision: 0.4985
Model as a Percentage: 49.8%
--------------------------------------------------
F1 Score: 0.5449
Model as a Percentage: 54.5%

Testing Config 3: RMSProp
Model compiled successfully with the following configuration:
Name: Config 3: RMSProp
Layers: [64, 32, 16]
Dropouts: [0.3, 0.3, 0.2]
Optimizer: RMSprop with learning rate 0.0005
Batch_size: 32
Initializer: glorot_uniform


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 91: early stopping
Restoring model weights from the end of the best epoch: 66.
Classfication Report for Neural Network - Config 3: RMSProp
Trained in: 41.66 seconds
--------------------------------------------------
Accuracy: 0.7802
Model as a Percentage: 78.0%
--------------------------------------------------
ROC AUC: 0.7735
Model as a Percentage: 77.4%
--------------------------------------------------
Recall: 0.5612
Model as a Percentage: 56.1%
--------------------------------------------------
Precision: 0.5155
Model as a Percentage: 51.5%
--------------------------------------------------
F1 Score: 0.5374
Model as a Percentage: 53.7%

Testing Config 4: Kaiming + Wider + Lower Dropout
Model compiled successfully with the following configuration:
Name: Config 4: Kaiming + Wider + Lower Dropout
Layers: [128, 64, 32]
Dropouts: [0.2, 0.2, 0.1]
Optimizer: RMSprop with learning rate 0.0010
Batch_size: 64
Initializer: he_normal


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 97: early stopping
Restoring model weights from the end of the best epoch: 72.
Classfication Report for Neural Network - Config 4: Kaiming + Wider + Lower Dropout
Trained in: 49.69 seconds
--------------------------------------------------
Accuracy: 0.7827
Model as a Percentage: 78.3%
--------------------------------------------------
ROC AUC: 0.7713
Model as a Percentage: 77.1%
--------------------------------------------------
Recall: 0.5524
Model as a Percentage: 55.2%
--------------------------------------------------
Precision: 0.5211
Model as a Percentage: 52.1%
--------------------------------------------------
F1 Score: 0.5363
Model as a Percentage: 53.6%

--------------------------------------------------


Tuning Configs created successfully!

--------------------------------------------------


# Key findings from Round 1 of Hyperparameter Tuning of NN

**The Goal**
Was to test if adjusting the optimizer, initialization, architecture, or dropout rates would improve the baseline neural network overall performance across all metrics to better predict default

The first list of configs consisted of 4 dictonaries with different hyperparameter variations

#### Findings: 
**Config 3 RMSProp** produced the most well rounded results among all configurations. Recall was slightly lower than the baseline but still the second highest recall. This indicated the model could have shifted from a aggressive to conservative approach in flagging defaults.

**Config2 SGD+Momentum** 
Performed nearly identical to the Baseline NN which utilized Adam optimizer. This suggests that the baseline model was well-suited to handle the current dataset 

**Wider Networks: Config 1 & 4**
Overall performance was not improve and suggest that they arent the most optimal for improving NN. The additional neurons failed to capture more meaningful patterns across the 23 features

**Kaiming Initialization** 
Paired with RSMprop acheived te second highest accuracy 0.7785 and precision 0.5119

### Takeaway 
An additional round of tuning should be applied but soley focusing on the top performing optimizer RMSProp and further tuning from there. Adjusting learning rates, batch size, initilizations, dropouts

# NN Tuning Comparison Table

In [18]:
# ============= Baseline vs Redesigned vs Tuned ===================

# check to see if tuning results were stored in pipeline results dictionary
for key in pipeline.results:
    print(key)    
    
tuning_data = [] 
for name, metrics in pipeline.results.items():
    if 'Config' in name:
        tuning_data.append({
            'Configuration': name,
            'Accuracy': round(metrics['accuracy'], 4),
            'ROC AUC': round(metrics['roc_auc'], 4),
            'Recall': round(metrics['recall'], 4),
            'Precision': round(metrics['precision'], 4),
            'F1-Score': round(metrics['f1-score'], 4),
            f"{'Training Time (SECONDS)':>25}": f"{metrics['training_time']:.2f}"
        })

# turn data into dataframe 
tuning_df = pd.DataFrame(tuning_data)
# sort by F1 Score 
tuning_df = tuning_df.sort_values(by='F1-Score', ascending=False)

print("=" * 50)
print("Hyperparmeter Tuning Results")
print("=" * 50)

# display comparison table with no index
print(tuning_df.to_string(index=False))
print("-" * 80)

Logistic Regression
Random Forest
LightGBM
Neural Network
Neural Network - Config 1: Wider Network
Neural Network - Config 2: SGD+Momentum
Neural Network - Config 3: RMSProp
Neural Network - Config 4: Kaiming + Wider + Lower Dropout
Hyperparmeter Tuning Results
                                             Configuration  Accuracy  ROC AUC  Recall  Precision  F1-Score   Training Time (SECONDS)
                   Neural Network - Config 2: SGD+Momentum    0.7717   0.7772  0.6007     0.4985    0.5449                     41.04
                        Neural Network - Config 3: RMSProp    0.7802   0.7735  0.5612     0.5155    0.5374                     41.66
                  Neural Network - Config 1: Wider Network    0.7613   0.7776  0.6081     0.4806    0.5369                     28.90
Neural Network - Config 4: Kaiming + Wider + Lower Dropout    0.7827   0.7713  0.5524     0.5211    0.5363                     49.69
-------------------------------------------------------------------------

In [19]:
# ======= Final Comparison ===========


baseline_redesign_tuned_comparison = {
    'Baseline': {
        'Accuracy': baseline_results['Neural Network']['accuracy'],
        'ROC AUC': baseline_results['Neural Network']['roc_auc'],
        'Recall': baseline_results['Neural Network']['recall'],
        'Precision': baseline_results['Neural Network']['precision'],
        'F1-Score': baseline_results['Neural Network']['f1-score']
    },
    'Redesigned': {
        'Accuracy': pipeline.results['Neural Network']['accuracy'],
        'ROC AUC': pipeline.results['Neural Network']['roc_auc'],
        'Recall': pipeline.results['Neural Network']['recall'],
        'Precision': pipeline.results['Neural Network']['precision'],
        'F1-Score': pipeline.results['Neural Network']['f1-score'],
    },
    'Tuned (RMSProp)': {
        'Accuracy': pipeline.results['Neural Network - Config 3: RMSProp']['accuracy'],
        'ROC AUC': pipeline.results['Neural Network - Config 3: RMSProp']['roc_auc'],
        'Recall': pipeline.results['Neural Network - Config 3: RMSProp']['recall'],
        'Precision': pipeline.results['Neural Network - Config 3: RMSProp']['precision'],
        'F1-Score': pipeline.results['Neural Network - Config 3: RMSProp']['f1-score']
    },
}

# create dataframe and use .T for transposing models as rows and metrics as columns 
final_nn_df = pd.DataFrame(baseline_redesign_tuned_comparison).T
final_nn_df.index.name = 'Model Comparison'

print("=" * 50)
print("Baseline -> Redesigned -> Best Tuned Comparison")
print("=" * 50)

# display comparison table with no index
print(final_nn_df.to_string())
print("-" * 80)

Baseline -> Redesigned -> Best Tuned Comparison
                  Accuracy   ROC AUC    Recall  Precision  F1-Score
Model Comparison                                                   
Baseline          0.766300  0.783900  0.605900   0.489100  0.541200
Redesigned        0.754667  0.778210  0.622711   0.470393  0.535939
Tuned (RMSProp)   0.780167  0.773514  0.561172   0.515478  0.537355
--------------------------------------------------------------------------------


# Tuning Analysis from Baseline -> Redesign -> Tuned Comparison

**Best Overall: Tuned RMSProp** 
Slight improvements and decrease but overall the model attributed that its more precise in all metrics, not just two or 3  and its flagging defaulters more often. It's balanced and can be a deployable model. 

The **baseline model** came in a tight lower 1st to 2nd place as it has higher F1 but under a different random seed. The redesign mitigated those flaws and ensure fair comparison across the environment. 

RMSProp improved accuracy by 2.5% and precision by 4.6% 

**Round 2 of configs** will refine around RMSprop findings and tweak its hyerparameters to see if some metrics that decrease will recover and improve

In [20]:

#======== Round 2 of Configs: only focusing on Best tuned RMSprop from Config group 1 =========


tuning_configs_two = [
    {
        'name': 'Config 5: RMSprop + Wider',
        'layers': [128,64,32],
        'dropouts': [0.3, 0.3, 0.2],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.0005),
        'batch_size': 64,
        'initializer': 'glorot_uniform'
    },
    {
        'name': 'Config 6: RMSprop + Lower Dropout',
        'layers': [64,32,16],
        'dropouts': [0.2, 0.2, 0.1],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.0005),
        'batch_size': 32,
        'initializer': 'glorot_uniform'
    },
    {
        'name': 'Config 7: RMSProp + Kaiming + Wider',
        'layers': [128,64,32],
        'dropouts': [0.2, 0.3, 0.2],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.0005),
        'batch_size': 32,
        'initializer': 'he_normal'
    },
    {
        'name': 'Config 8: RMSProp + Deep Kaiming + Small Batch',
        'layers': [256,128,64],
        'dropouts': [0.4, 0.3, 0.2],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.000025),
        'batch_size': 16,
        'initializer': 'he_normal'
    },
        {
        'name': 'Config 9: RMSProp + Small Batch',
        'layers': [128,64,32],
        'dropouts': [0.2, 0.2, 0.2],
        'optimizer': tf.keras.optimizers.RMSprop(learning_rate=0.0001),
        'batch_size': 16,
        'initializer': 'he_normal'
    },
]


print("-" * 50)
print("Tuning Configurations initialized successfully!\n")
print("-" * 50)

--------------------------------------------------
Tuning Configurations initialized successfully!

--------------------------------------------------


In [21]:
# loop through each config and train a new model with one of the specified hyperparameter configurations from the tuning_configs list
for config in tuning_configs_two:
    print("=" * 50)
    print(f"Testing {config['name']}")
    print("=" * 50)

    model = Sequential([
    Dense(config['layers'][0], activation='relu', kernel_initializer=config['initializer'], name='hidden_layer_1', input_shape=(23,)),
    Dropout(config['dropouts'][0], name='dropout_1'),
    
    Dense(config['layers'][1], activation='relu', kernel_initializer=config['initializer'], name='hidden_layer_2'),
    Dropout(config['dropouts'][1], name='dropout_2'),
    
    Dense(config['layers'][2], activation='relu', kernel_initializer=config['initializer'], name='hidden_layer_3'),
    Dropout(config['dropouts'][2], name='dropout_3'),
    
    Dense(1, activation='sigmoid', name='output_layer')
])
    
    # ========= COMPILE THE MODEL ========= 
    model.compile(
    optimizer=config['optimizer'],
    loss='binary_crossentropy',
    metrics=['accuracy']
)

    print('Model compiled successfully with the following configuration:')
    for key, value in config.items():
        if key != 'optimizer':
            print(f"{key.capitalize()}: {value}")
        else:
            print(f"Optimizer: {type(value).__name__} with learning rate {value.learning_rate.numpy():.4f}")
        

    # ========= TRAIN THE MODEL ========= 
    pipeline.train_model(model, f"Neural Network - {config['name']}", scale=True)
    
print("-" * 50)
print("\n\nTuning Configs created successfully!\n")
print("-" * 50)

Testing Config 5: RMSprop + Wider
Model compiled successfully with the following configuration:
Name: Config 5: RMSprop + Wider
Layers: [128, 64, 32]
Dropouts: [0.3, 0.3, 0.2]
Optimizer: RMSprop with learning rate 0.0005
Batch_size: 64
Initializer: glorot_uniform


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 48: early stopping
Restoring model weights from the end of the best epoch: 23.
Classfication Report for Neural Network - Config 5: RMSprop + Wider
Trained in: 24.99 seconds
--------------------------------------------------
Accuracy: 0.7683
Model as a Percentage: 76.8%
--------------------------------------------------
ROC AUC: 0.7695
Model as a Percentage: 76.9%
--------------------------------------------------
Recall: 0.5773
Model as a Percentage: 57.7%
--------------------------------------------------
Precision: 0.4922
Model as a Percentage: 49.2%
--------------------------------------------------
F1 Score: 0.5314
Model as a Percentage: 53.1%

Testing Config 6: RMSprop + Lower Dropout
Model compiled successfully with the following configuration:
Name: Config 6: RMSprop + Lower Dropout
Layers: [64, 32, 16]
Dropouts: [0.2, 0.2, 0.1]
Optimizer: RMSprop with learning rate 0.0005
Batch_size: 32
Initializer: glorot_uniform


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 56: early stopping
Restoring model weights from the end of the best epoch: 31.
Classfication Report for Neural Network - Config 6: RMSprop + Lower Dropout
Trained in: 25.83 seconds
--------------------------------------------------
Accuracy: 0.7782
Model as a Percentage: 77.8%
--------------------------------------------------
ROC AUC: 0.7755
Model as a Percentage: 77.6%
--------------------------------------------------
Recall: 0.5722
Model as a Percentage: 57.2%
--------------------------------------------------
Precision: 0.5111
Model as a Percentage: 51.1%
--------------------------------------------------
F1 Score: 0.5399
Model as a Percentage: 54.0%

Testing Config 7: RMSProp + Kaiming + Wider
Model compiled successfully with the following configuration:
Name: Config 7: RMSProp + Kaiming + Wider
Layers: [128, 64, 32]
Dropouts: [0.2, 0.3, 0.2]
Optimizer: RMSprop with learning rate 0.0005
Batch_size: 32
Initializer: he_normal


...

Epoch 80: early stopping
Restoring model weights from the end of the best epoch: 55.
Classfication Report for Neural Network - Config 7: RMSProp + Kaiming + Wider
Trained in: 42.89 seconds
--------------------------------------------------
Accuracy: 0.7908
Model as a Percentage: 79.1%
--------------------------------------------------
ROC AUC: 0.7747
Model as a Percentage: 77.5%
--------------------------------------------------
Recall: 0.5414
Model as a Percentage: 54.1%
--------------------------------------------------
Precision: 0.5402
Model as a Percentage: 54.0%
--------------------------------------------------
F1 Score: 0.5408
Model as a Percentage: 54.1%

Testing Config 8: RMSProp + Deep Kaiming + Small Batch
Model compiled successfully with the following configuration:
Name: Config 8: RMSProp + Deep Kaiming + Small Batch
Layers: [256, 128, 64]
Dropouts: [0.4, 0.3, 0.2]
Optimizer: RMSprop with learning rate 0.0000
Batch_size: 16
Initializer: he_normal


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 28: early stopping
Restoring model weights from the end of the best epoch: 3.
Classfication Report for Neural Network - Config 8: RMSProp + Deep Kaiming + Small Batch
Trained in: 19.01 seconds
--------------------------------------------------
Accuracy: 0.6783
Model as a Percentage: 67.8%
--------------------------------------------------
ROC AUC: 0.7099
Model as a Percentage: 71.0%
--------------------------------------------------
Recall: 0.6095
Model as a Percentage: 61.0%
--------------------------------------------------
Precision: 0.3733
Model as a Percentage: 37.3%
--------------------------------------------------
F1 Score: 0.4630
Model as a Percentage: 46.3%

Testing Config 9: RMSProp + Small Batch
Model compiled successfully with the following configuration:
Name: Config 9: RMSProp + Small Batch
Layers: [128, 64, 32]
Dropouts: [0.2, 0.2, 0.2]
Optimizer: RMSprop with learning rate 0.0001
Batch_size: 16
Initializer: he_normal


/Users/djmike20/Documents/NU - Applied AI/AAI6610 - Applied Machine Learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 99: early stopping
Restoring model weights from the end of the best epoch: 74.
Classfication Report for Neural Network - Config 9: RMSProp + Small Batch
Trained in: 57.19 seconds
--------------------------------------------------
Accuracy: 0.7810
Model as a Percentage: 78.1%
--------------------------------------------------
ROC AUC: 0.7707
Model as a Percentage: 77.1%
--------------------------------------------------
Recall: 0.5678
Model as a Percentage: 56.8%
--------------------------------------------------
Precision: 0.5170
Model as a Percentage: 51.7%
--------------------------------------------------
F1 Score: 0.5412
Model as a Percentage: 54.1%

--------------------------------------------------


Tuning Configs created successfully!

--------------------------------------------------


In [22]:
# ====== create config two comparison table ======


tuning_r2_data = [] 
for name, metrics in pipeline.results.items():
    if 'Config 5' in name or 'Config 6' in name or 'Config 7' in name or 'Config 8' in name or 'Config 9' in name:
        tuning_r2_data.append({
            'Configuration': name,
            'Accuracy': round(metrics['accuracy'], 4),
            'ROC AUC': round(metrics['roc_auc'], 4),
            'Recall': round(metrics['recall'], 4),
            'Precision': round(metrics['precision'], 4),
            'F1-Score': round(metrics['f1-score'], 4),
            f"{'Training Time (SECONDS)':>25}": f"{metrics['training_time']:.2f}"
        })

# turn data into dataframe 
tuning_df = pd.DataFrame(tuning_r2_data)
# sort by F1 Score 
tuning_df = tuning_df.sort_values(by='F1-Score', ascending=False)

print("=" * 50)
print("Config Hyperparmeter Tuning Results")
print("=" * 50)

# display comparison table with no index
print(tuning_df.to_string(index=False))
print("-" * 80)

Config Hyperparmeter Tuning Results
                                                  Configuration  Accuracy  ROC AUC  Recall  Precision  F1-Score   Training Time (SECONDS)
               Neural Network - Config 9: RMSProp + Small Batch    0.7810   0.7707  0.5678     0.5170    0.5412                     57.19
           Neural Network - Config 7: RMSProp + Kaiming + Wider    0.7908   0.7747  0.5414     0.5402    0.5408                     42.89
             Neural Network - Config 6: RMSprop + Lower Dropout    0.7782   0.7755  0.5722     0.5111    0.5399                     25.83
                     Neural Network - Config 5: RMSprop + Wider    0.7683   0.7695  0.5773     0.4922    0.5314                     24.99
Neural Network - Config 8: RMSProp + Deep Kaiming + Small Batch    0.6783   0.7099  0.6095     0.3733    0.4630                     19.01
--------------------------------------------------------------------------------


# SAVING THE BEST MODEL FOR HUGGING FACE SPACES

In [23]:
''' 
The goal of this section is to implement CI/CD pipelines for the credit risk prediction model: LightGBM and its feature names to get ready for deployment to HuggingFace Space.

I selected LightGBM as the model for deployment because it performed the best across all metrics even with the improvement of adding class weights. 
The intention was to verfiy if neural networks would outperform the tree based models with addition to tuning but improvement was minimal
- With LighGBM being lightweight, fast ,and high performing and can run off CPUs, makes it a prime candidate for deployment 

- Results
    - Bestt F1 Score: (0.5519) among all balanced class weights
    - Best Recall: (0.6352 ) core for detecting defaults within credit risk
    - Inference training time of 0.78 seconds with handling the class imbalance, well suited for CPU Hugging Face Space Free tier deployment 
    
- No Scaling required for tree based models 
- The goal
    - Save the LightGBM model itself using joblib or pickle 
    - Save the feature names to ensure correct order of the 23 listed features during inference and deployment.
        - using Gradio to map user inputs and better handling during inferencing and deployment 
'''

import joblib 


# Pull the LightGBM model from the pipeline's dictionary 
lgbm_trained_model_deployment = pipeline.models["LightGBM"]['model']

# save the model using joblib.dump 
joblib.dump(lgbm_trained_model_deployment, 'lgbm_credit_risk_model.joblib')

# save the features (df_features) using joblib.dump
joblib.dump(pipeline.df_features, 'lgbm_credit_risk_features.joblib')


print("Model saved successfully using joblib!")
print("-" * 50)
print("Model saved: lgbm_credit_risk_model.joblib")
print("Features saved: lgbm_credit_risk_features.joblib")
print(f'Number of Features: {len(pipeline.df_features)}')


Model saved successfully using joblib!
--------------------------------------------------
Model saved: lgbm_credit_risk_model.joblib
Features saved: lgbm_credit_risk_features.joblib
Number of Features: 23
